# Step 1: 에이전트 루프 + 스트리밍 — 가장 핵심적인 패턴

## 📋 학습 목표

- Claude Code의 **핵심 에이전트 루프** (`while(true)` 패턴)를 이해합니다
- **AsyncGenerator** 패턴으로 스트리밍 응답을 처리하는 방법을 배웁니다
- Python으로 Anthropic/OpenAI를 모두 지원하는 **LLM 추상화 레이어**와 **에이전트 루프**를 직접 구현합니다

> **에이전트(Agent)란?**
> LLM이 텍스트만 생성하는 게 아니라, 도구를 호출하고 그 결과를 받아 다시 생각하는 과정을 반복하는 시스템입니다.
> 이 "반복"을 구현하는 것이 바로 에이전트 루프입니다.

---

## 🔍 Claude Code 소스 분석

### 1-1. 에이전트 루프의 전체 그림

Claude Code의 에이전트 루프는 `src/query.ts`에 있습니다. 512,000줄이 넘는 코드베이스의 심장부이지만, 핵심 구조는 놀랍도록 단순합니다:

```
사용자 입력 → [while 루프 시작]
                  ↓
            LLM API 호출 (스트리밍)
                  ↓
            응답에 tool_use가 있나?
              ├─ 없음 → return (루프 종료)
              └─ 있음 → 도구 실행 → 결과를 메시지에 추가 → [루프 반복]
```

이것이 전부입니다. 모든 복잡한 기능(컨텍스트 압축, 권한 체크, 서브에이전트 등)은 이 루프 안의 각 단계에 플러그인 형태로 끼워 넣어진 것입니다.

### 1-2. State — 불변 상태 객체 (query.ts:204-217)

Claude Code는 루프 상태를 하나의 `State` 객체로 관리합니다:

```typescript
// src/query.ts:204-217

// Mutable state carried between loop iterations
type State = {
  messages: Message[]                    // 대화 히스토리
  toolUseContext: ToolUseContext          // 도구 실행에 필요한 환경
  autoCompactTracking: ... | undefined   // 자동 압축 상태
  maxOutputTokensRecoveryCount: number   // 출력 토큰 초과 복구 횟수
  hasAttemptedReactiveCompact: boolean   // reactive compact 시도 여부
  turnCount: number                      // 현재 반복 횟수
  transition: Continue | undefined       // 이전 반복의 종료 이유
}
```

**TypeScript 문법 참고:**
- `type State = { ... }` → Python의 `@dataclass` 또는 `TypedDict`에 해당
- `Message[]` → Python의 `list[Message]`
- `... | undefined` → Python의 `... | None`

**설계 포인트: 왜 불변 상태인가?**

매 반복 끝에서 `state = { ...새로운값들 }` (spread 연산자)로 새 객체를 만듭니다. 이전 state를 수정하지 않기 때문에:
- 에러 발생 시 이전 상태로 쉽게 돌아갈 수 있습니다
- `transition` 필드로 "왜 다음 반복으로 넘어갔는지" 디버깅할 수 있습니다
- 병렬 실행 시 상태 충돌이 발생하지 않습니다

### 1-3. queryLoop() — while(true)의 실제 코드 (query.ts:241-307, 1715-1728)

```typescript
// src/query.ts:241-250 — 함수 시그니처
async function* queryLoop(            // async function* = AsyncGenerator
  params: QueryParams,
  consumedCommandUuids: string[],
): AsyncGenerator<StreamEvent | Message | ...>  // yield하는 타입들
{
  // 초기 State 생성 (268-279)
  let state: State = {
    messages: params.messages,
    toolUseContext: params.toolUseContext,
    turnCount: 1,
    maxOutputTokensRecoveryCount: 0,
    hasAttemptedReactiveCompact: false,
    transition: undefined,       // 첫 반복은 전이 이유 없음
    // ...
  }

  // ★ 핵심 루프 (307)
  while (true) {
    // 매 반복 시작: State를 destructure
    let { toolUseContext } = state
    const { messages, turnCount, ... } = state

    // ... (컨텍스트 준비, 압축 확인 등 — 이후 Step에서 다룸)

    // API 호출
    let needsFollowUp = false           // (558) 도구 호출 여부 플래그
    for await (const event of stream) { // 스트리밍 응답 처리
      // tool_use 블록이 발견되면 needsFollowUp = true
    }

    // ★ 루프 종료 판단 (1062)
    if (!needsFollowUp) {
      return { reason: 'completed' }    // 도구 호출 없음 → 종료
    }

    // max_turns 확인 (1704-1712)
    if (maxTurns && nextTurnCount > maxTurns) {
      return { reason: 'max_turns' }    // 최대 반복 도달 → 종료
    }

    // ★ 다음 턴 State 생성 (1715-1728) — 불변 전환
    const next: State = {
      messages: [...messagesForQuery, ...assistantMessages, ...toolResults],
      toolUseContext: toolUseContextWithQueryTracking,
      turnCount: nextTurnCount,
      maxOutputTokensRecoveryCount: 0,    // 매 턴마다 리셋
      hasAttemptedReactiveCompact: false,  // 매 턴마다 리셋
      transition: { reason: 'next_turn' },
      // ...
    }
    state = next      // 새 State로 교체
  } // while (true)
}
```

**TypeScript 문법 참고:**
- `async function*` → Python의 `async def` + `yield` (async generator)
- `yield*` → Python에는 직접 대응 없음. `async for event in sub_generator: yield event`로 구현
- `for await (... of stream)` → Python의 `async for ... in stream`
- `{ ...obj, key: value }` (spread) → Python의 `dataclasses.replace(obj, key=value)`

### 1-4. AsyncGenerator 패턴 — 왜 generator인가?

Claude Code가 일반 `async function`이 아닌 `async function*` (AsyncGenerator)를 사용하는 이유는 **스트리밍** 때문입니다.

일반 함수라면 모든 도구 실행이 끝날 때까지 아무것도 반환하지 못합니다. 하지만 generator는 실행 도중에 `yield`로 중간 결과를 내보낼 수 있습니다:

```
[Turn 1]
  yield StreamEvent("message_start")
  yield StreamEvent(text="파일을")      ← 사용자가 실시간으로 텍스트를 봄
  yield StreamEvent(text=" 읽겠습니다")
  yield StreamEvent(tool_use: "Read")
  // 도구 실행...
  yield ToolResult("파일 내용: ...")

[Turn 2]
  yield StreamEvent(text="분석 결과는...")
  yield StreamEvent("message_end")

return { reason: "completed" }
```

이 패턴 덕분에 Claude Code의 터미널 UI는 **LLM이 생각하는 동안** 실시간으로 텍스트를 표시하고, **도구가 실행되는 동안** 프로그레스 바를 보여줄 수 있습니다.

---

## 🏗️ Python으로 구현하기

이제 위 패턴을 Python으로 구현합니다. 두 개의 모듈을 만듭니다:

1. **`mini_claude/llm_client.py`** — LLM API 추상화 레이어 (Anthropic + OpenAI)
2. **`mini_claude/agent_loop.py`** — 핵심 while(true) 에이전트 루프

### 구현 1: LLM 클라이언트 추상화

Claude Code는 Anthropic API만 사용하지만(`src/services/api/claude.ts`), 이 워크샵에서는 학습 목적으로 두 API를 모두 지원합니다. 핵심은 **도구 호출의 통합 표현**입니다:

- Anthropic: `content` 배열에 `tool_use` 블록이 포함됨
- OpenAI: `tool_calls` 배열이 별도로 존재

이 차이를 `ToolCall(id, name, input)` 하나의 타입으로 통합합니다.

In [ ]:
# llm_client.py의 핵심 구조를 살펴봅시다

# 1. 통합 응답 타입
from dataclasses import dataclass, field
from typing import Any

@dataclass
class ToolCall:
    """LLM이 요청한 도구 호출 — Anthropic의 tool_use 블록, OpenAI의 function call 모두를 표현"""
    id: str       # 고유 ID (도구 결과를 돌려줄 때 매칭에 사용)
    name: str     # 도구 이름 (예: "Read", "Bash")
    input: dict   # 도구에 전달할 인자 (예: {"file_path": "/tmp/test.py"})

@dataclass
class LLMResponse:
    """LLM 응답 — 텍스트와 도구 호출을 모두 포함할 수 있음"""
    text: str = ""
    tool_calls: list[ToolCall] = field(default_factory=list)
    input_tokens: int = 0
    output_tokens: int = 0

    @property
    def has_tool_calls(self) -> bool:
        """Claude Code의 needsFollowUp (query.ts:558)에 대응"""
        return len(self.tool_calls) > 0

# 2. 이 통합 타입 덕분에, 에이전트 루프는 API에 무관하게 동일한 코드로 동작합니다
response = LLMResponse(
    text="파일을 읽겠습니다.",
    tool_calls=[ToolCall(id="call_1", name="Read", input={"file_path": "README.md"})],
)
print(f"텍스트: {response.text}")
print(f"도구 호출 있음? {response.has_tool_calls}")
print(f"호출할 도구: {response.tool_calls[0].name}({response.tool_calls[0].input})")

### 구현 2: 불변 State

Claude Code의 `State` 타입(query.ts:204)을 Python `frozen dataclass`로 구현합니다.
`frozen=True`이면 필드 값을 직접 변경할 수 없고, `replace()`로 새 객체를 만들어야 합니다.

In [ ]:
from dataclasses import dataclass, replace

@dataclass(frozen=True)
class State:
    """에이전트 루프의 불변 상태 — Claude Code의 State (query.ts:204)"""
    messages: tuple[dict, ...]   # tuple = 불변 리스트
    turn_count: int = 1
    transition: str = ""         # "", "tool_use", "completed", "max_turns"

    def next_turn(self, new_messages: list[dict], transition: str = "tool_use") -> "State":
        """다음 턴을 위한 새 State — query.ts:1715-1728"""
        return replace(
            self,
            messages=tuple(list(self.messages) + new_messages),
            turn_count=self.turn_count + 1,
            transition=transition,
        )

# 사용 예시
state1 = State(messages=({"role": "user", "content": "안녕"},))
print(f"Turn {state1.turn_count}: 메시지 {len(state1.messages)}개")

# 새 메시지 추가 → 새 State 생성 (state1은 변하지 않음)
state2 = state1.next_turn([{"role": "assistant", "content": "반갑습니다!"}])
print(f"Turn {state2.turn_count}: 메시지 {len(state2.messages)}개, transition={state2.transition}")
print(f"state1은 그대로: 메시지 {len(state1.messages)}개")  # 불변 확인

### 구현 3: 에이전트 루프 핵심

이제 핵심인 `while True` 루프를 구현합니다. `async def` + `yield`로 AsyncGenerator 패턴을 사용합니다.

아래 코드는 `mini_claude/agent_loop.py`의 핵심 부분을 Notebook에서 직접 확인할 수 있게 옮긴 것입니다.

In [ ]:
import asyncio
from dataclasses import dataclass, replace, field
from typing import AsyncIterator, Any, Callable, Awaitable

# --- 앞서 정의한 타입들을 다시 가져옵니다 (실제로는 mini_claude에서 import) ---

@dataclass
class AgentEvent:
    """에이전트 루프가 yield하는 이벤트"""
    type: str     # "text", "tool_call", "tool_result", "turn_start", "turn_end", "done"
    text: str = ""
    tool_call: Any = None
    tool_result: str = ""
    state: Any = None
    response: Any = None


async def agent_loop(
    *,
    client,                      # LLMClient 인스턴스
    messages: list[dict],        # 초기 대화 히스토리
    system: str = "",            # 시스템 프롬프트
    tools: list[dict] | None = None,
    tool_executor=None,          # (name, input) -> result 콜백
    max_turns: int = 20,
) -> AsyncIterator[AgentEvent]:
    """
    Claude Code의 queryLoop() (query.ts:307)에 대응하는 에이전트 루프.
    
    핵심 구조:
    1. while True:
    2.   response = await client.query(messages)
    3.   if not response.has_tool_calls: return     ← 루프 종료
    4.   results = await execute_tools(response)
    5.   messages += [response, results]             ← 다음 반복
    """
    from mini_claude.llm_client import LLMResponse, tool_result_message
    from mini_claude.agent_loop import State

    state = State(messages=tuple(messages))

    # ★ while (true) — query.ts:307
    while True:
        yield AgentEvent(type="turn_start", state=state)

        # 1. LLM API 호출
        response = await client.query(
            messages=list(state.messages),
            system=system, tools=tools, max_tokens=4096,
        )

        if response.text:
            yield AgentEvent(type="text", text=response.text, response=response)

        # 2. tool_use가 없으면 → 종료 (query.ts:1062)
        if not response.has_tool_calls:
            yield AgentEvent(type="done", state=replace(state, transition="completed"))
            return

        # 3. 도구 실행
        new_messages = []
        # assistant 메시지를 히스토리에 추가
        new_messages.append({
            "role": "assistant", "text": response.text,
            "tool_calls": [{"id": tc.id, "name": tc.name, "input": tc.input}
                           for tc in response.tool_calls],
        })

        for tc in response.tool_calls:
            yield AgentEvent(type="tool_call", tool_call=tc)
            result = await tool_executor(tc.name, tc.input) if tool_executor else "N/A"
            yield AgentEvent(type="tool_result", tool_call=tc, tool_result=result)
            new_messages.append(tool_result_message(tc.id, result))

        # 4. max_turns 확인 (query.ts:1704)
        if state.turn_count >= max_turns:
            yield AgentEvent(type="done", state=state.next_turn(new_messages, "max_turns"))
            return

        # 5. 다음 턴으로 — 새 State 생성 (query.ts:1715)
        state = state.next_turn(new_messages, transition="tool_use")
        yield AgentEvent(type="turn_end", state=state)

print("✅ 에이전트 루프 정의 완료 — 이 코드는 mini_claude/agent_loop.py에도 저장되어 있습니다")

---

## 🧪 실습: 실제 API로 에이전트 루프 실행해보기

이제 진짜 LLM API를 호출하여 에이전트 루프를 실행합니다. 간단한 도구(현재 시간 조회, 계산기)를 등록하고, LLM이 도구를 호출하는 전체 과정을 확인합니다.

> **API 키 설정**: `ANTHROPIC_API_KEY` 또는 `OPENAI_API_KEY` 환경변수가 설정되어 있어야 합니다.

In [ ]:
import sys, os

# mini_claude 패키지를 import할 수 있도록 경로 추가
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
sys.path.insert(0, ".")

from mini_claude.llm_client import create_client, tool_to_anthropic_schema, tool_to_openai_schema

# 클라이언트 생성 (환경변수에서 자동 감지)
client = create_client()
print(f"✅ 클라이언트 생성 완료: {type(client).__name__}")

In [ ]:
import datetime

# --- 도구 정의 ---
# Step 2에서 정식 도구 시스템을 만들기 전에, 여기서는 딕셔너리로 간단하게 정의합니다.

# 1. 도구 스키마 (LLM에게 알려줄 도구 설명)
#    Anthropic과 OpenAI의 스키마 형식이 다르므로, 클라이언트 타입에 따라 변환합니다.
from mini_claude.llm_client import AnthropicClient

if isinstance(client, AnthropicClient):
    tools_schema = [
        tool_to_anthropic_schema(
            "get_current_time",
            "현재 시간을 반환합니다. timezone 인자로 시간대를 지정할 수 있습니다.",
            {
                "type": "object",
                "properties": {
                    "timezone": {
                        "type": "string",
                        "description": "시간대 (예: 'Asia/Seoul', 'UTC'). 기본값: UTC"
                    }
                },
                "required": []
            }
        ),
        tool_to_anthropic_schema(
            "calculate",
            "수학 표현식을 계산합니다.",
            {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "계산할 수학 표현식 (예: '2 + 3 * 4')"
                    }
                },
                "required": ["expression"]
            }
        ),
    ]
else:
    tools_schema = [
        tool_to_openai_schema(
            "get_current_time",
            "현재 시간을 반환합니다. timezone 인자로 시간대를 지정할 수 있습니다.",
            {
                "type": "object",
                "properties": {
                    "timezone": {
                        "type": "string",
                        "description": "시간대 (예: 'Asia/Seoul', 'UTC'). 기본값: UTC"
                    }
                },
                "required": []
            }
        ),
        tool_to_openai_schema(
            "calculate",
            "수학 표현식을 계산합니다.",
            {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "계산할 수학 표현식 (예: '2 + 3 * 4')"
                    }
                },
                "required": ["expression"]
            }
        ),
    ]

# 2. 도구 실행기 (실제로 도구를 실행하는 함수)
async def execute_tool(name: str, args: dict) -> str:
    if name == "get_current_time":
        tz = args.get("timezone", "UTC")
        now = datetime.datetime.now(datetime.timezone.utc)
        return f"현재 시간 ({tz}): {now.isoformat()}"
    elif name == "calculate":
        expr = args.get("expression", "")
        try:
            # 주의: 실제 제품에서는 eval을 사용하면 안 됩니다 (보안 위험)
            result = eval(expr, {"__builtins__": {}})
            return f"{expr} = {result}"
        except Exception as e:
            return f"계산 오류: {e}"
    else:
        return f"알 수 없는 도구: {name}"

print(f"✅ 도구 {len(tools_schema)}개 정의 완료: get_current_time, calculate")

### 실행! — mini_claude의 에이전트 루프로 실제 대화

이제 `mini_claude` 패키지의 `run_agent()`를 사용해 에이전트를 실행합니다.
`verbose=True`로 도구 호출 과정을 관찰할 수 있습니다.

In [ ]:
from mini_claude.agent_loop import run_agent

# 에이전트 실행 — 도구를 호출하는 질문을 해봅시다
result = await run_agent(
    client=client,
    prompt="지금 몇 시야? 그리고 123 * 456 + 789의 결과를 알려줘.",
    tools=tools_schema,
    tool_executor=execute_tool,
    verbose=True,  # 도구 호출 과정을 출력
)

print("\n" + "=" * 60)
print("📝 최종 응답:")
print(result)

### AsyncGenerator 직접 소비하기

`run_agent()`는 편의 함수이고, 내부적으로 `agent_loop()` AsyncGenerator를 소비합니다.
직접 `async for`로 이벤트를 하나씩 처리하면, Claude Code처럼 실시간 UI를 만들 수 있습니다.

In [ ]:
from mini_claude.agent_loop import agent_loop as real_agent_loop

# AsyncGenerator를 직접 소비하여 이벤트별로 처리
messages = [{"role": "user", "content": "현재 시간을 알려주고, 그 시간의 분(minute)을 제곱해줘."}]

async for event in real_agent_loop(
    client=client,
    messages=messages,
    system="당신은 도구를 사용하여 사용자를 돕는 AI 어시스턴트입니다. 한국어로 답변하세요.",
    tools=tools_schema,
    tool_executor=execute_tool,
    max_turns=10,
):
    # 이벤트 타입별로 다르게 처리 — Claude Code의 UI 렌더링에 대응
    if event.type == "turn_start":
        print(f"\n{'─' * 40} Turn {event.state.turn_count} {'─' * 40}")
    elif event.type == "text":
        print(f"💬 {event.text[:200]}")
    elif event.type == "tool_call":
        tc = event.tool_call
        print(f"🔧 도구 호출: {tc.name}({tc.input})")
    elif event.type == "tool_result":
        print(f"📋 결과: {event.tool_result}")
    elif event.type == "done":
        print(f"\n✅ 완료 (transition: {event.state.transition})")

---

## 💡 핵심 설계 원칙 정리

### 1. while(true) + 종료 조건 = 에이전트 루프
에이전트의 핵심은 무한 루프입니다. LLM이 도구를 호출하면 계속, 텍스트만 반환하면 종료. 이 단순한 패턴이 파일 읽기, 코드 작성, 테스트 실행까지 모든 복잡한 작업을 가능하게 합니다.

### 2. 불변 상태 전환
매 반복마다 이전 State를 복사해서 새 State를 만듭니다. 이전 상태를 수정하지 않으므로 에러 복구와 디버깅이 쉽습니다. Python에서는 `frozen dataclass` + `replace()`로 구현합니다.

### 3. AsyncGenerator로 스트리밍
`async def` + `yield`로 구현한 에이전트 루프는 호출자가 `async for`로 이벤트를 실시간 소비할 수 있게 합니다. 이것이 Claude Code가 터미널에서 실시간 텍스트 출력과 프로그레스 표시를 하는 방법입니다.

### 4. API 추상화
Claude Code는 Anthropic 전용이지만, 도구 호출의 통합 표현(`ToolCall`)을 만들면 API에 무관하게 에이전트 로직을 재사용할 수 있습니다.

---

## ➡️ 다음 Step 미리보기

현재 도구 정의가 딕셔너리와 `if/elif` 분기로 되어 있어 확장이 어렵습니다.

**Step 2**에서는 Claude Code의 `Tool.ts`를 분석하여:
- **Tool 프로토콜** — 모든 도구가 따르는 통일된 인터페이스
- **build_tool() 팩토리** — 안전한 기본값으로 도구 생성을 단순화
- **ToolRegistry** — 도구를 이름으로 찾고, API 스키마를 자동 생성

이 세 가지를 Python으로 구현합니다.